<a href="https://colab.research.google.com/github/pablo-arantes/spde-workshop-ai-protein-design/blob/main/Notebook2_PyRosetta_Physics_Filters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyRosetta Physics Filters — Post-Design Validation

This notebook takes the ZIP output from the **RFdiffusion3 Design Workshop** and applies
Rosetta-level physics validation: **FastRelax** for structural optimization, followed by
energy-based **filters** (ddG, SAP score, contact molecular surface, SASA, buried
unsatisfied H-bonds) to identify the most promising binder candidates.

In [1]:
#@title **0.1 — Install Dependencies** { display-mode: "form" }
#@markdown > Installs PyRosetta (academic license via `pyrosetta-installer`),
#@markdown > py3Dmol for 3D visualization, and supporting packages.
#@markdown >
#@markdown > **Note:** PyRosetta requires an academic license. If `pyrosetta_installer`
#@markdown > prompts for credentials, use your RosettaCommons username/password.

import subprocess, sys, importlib

def _is_installed(pkg):
    try:
        importlib.import_module(pkg)
        return True
    except ImportError:
        return False

# ── Core scientific stack ──
!pip install -q pandas numpy matplotlib py3Dmol biopython

# ── PyRosetta ──
if not _is_installed("pyrosetta"):
    print("📥 Installing PyRosetta (this takes a few minutes)...")
    !pip install -q pyrosetta-installer
    import pyrosetta_installer
    pyrosetta_installer.install_pyrosetta()
else:
    print("✅ PyRosetta already installed")

import pyrosetta
print(f"✅ PyRosetta ready")
print("✅ All dependencies installed")

✅ PyRosetta already installed
✅ PyRosetta ready
✅ All dependencies installed


In [9]:
#@title **0.2 — Upload Design ZIP & Extract Relaxed Structures** { display-mode: "form" }
#@markdown > Upload the `*_designs.zip` file from the **RFdiffusion3 Workshop** notebook.
#@markdown > The code extracts structures from `03_predictions/` and selects only
#@markdown > the **relaxed** PDBs (files containing `_relaxed_` in their name).
#@markdown > If no relaxed structures are found, all PDBs are used instead.

import os, glob, shutil, zipfile
from google.colab import files
from pathlib import Path

WORK_DIR = "/content/pyrosetta_filters"
INPUT_DIR = os.path.join(WORK_DIR, "input_pdbs")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
for d in [WORK_DIR, INPUT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("📤 Upload the designs ZIP from the previous workshop notebook:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded. Please upload the *_designs.zip file.")

zip_filename = list(uploaded.keys())[0]
zip_path = os.path.join(WORK_DIR, zip_filename)

# Move uploaded file to working directory
if not os.path.exists(zip_path):
    shutil.move(zip_filename, zip_path)

# Extract
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(WORK_DIR)
    all_members = zf.namelist()

print(f"\n📦 Extracted {len(all_members)} files from {zip_filename}")

# Find predictions
prediction_pdbs = sorted(
    glob.glob(os.path.join(WORK_DIR, "03_predictions", "*.pdb"))
)
print(f"   Found {len(prediction_pdbs)} PDBs in 03_predictions/")

# Prefer relaxed structures
relaxed_pdbs = [p for p in prediction_pdbs if "_relaxed_" in os.path.basename(p)]
if relaxed_pdbs:
    selected_pdbs = relaxed_pdbs
    print(f"   ✅ Selected {len(relaxed_pdbs)} relaxed structures")
else:
    selected_pdbs = prediction_pdbs
    print(f"   ⚠️  No relaxed structures found — using all {len(prediction_pdbs)} PDBs")

# Copy selected PDBs to input directory
for src in selected_pdbs:
    shutil.copy2(src, INPUT_DIR)

input_pdb_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdb")))
print(f"\n✅ {len(input_pdb_files)} PDBs ready in {INPUT_DIR}")
for p in input_pdb_files:
    print(f"   • {os.path.basename(p)}")

📤 Upload the designs ZIP from the previous workshop notebook:


Saving FPR2_designs.zip to FPR2_designs (1).zip

📦 Extracted 42 files from FPR2_designs (1).zip
   Found 32 PDBs in 03_predictions/
   ✅ Selected 16 relaxed structures

✅ 16 PDBs ready in /content/pyrosetta_filters/input_pdbs
   • inputs_fpr2_binder_0_model_0.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_0.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_0.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_1.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_1.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb
   • inputs_fpr2_binder_0_model_1.cif_b

---
## 1. Stage 1: FastRelax — Structural Optimization

Before computing physics-based metrics, each predicted complex is **relaxed** with
Rosetta’s FastRelax protocol. This removes steric clashes introduced by ColabFold,
optimizes side-chain packing, and moves the structure into a local energy minimum
so that downstream energy calculations (ddG, SASA, etc.) are physically meaningful.

**Key settings:**
- `standard_repeats = 15` — more repeats for better convergence
- `constrain_relax_to_start_coords = True` — keeps the backbone close to the AF2 prediction
- Binding energy = E(bound) – E(unbound)

In [10]:
#@title **1.0 — FastRelax Configuration** { display-mode: "form" }

#@markdown ### Chain Assignments
#@markdown Specify which chain is the binder (peptide) and which is the target.
peptide_chain = "B"  #@param ["A", "B", "C", "D", "E", "F"]
target_chain  = "A"  #@param ["A", "B", "C", "D", "E", "F"]

#@markdown ### FastRelax Repeats
fastrelax_repeats = 1  #@param {type:"slider", min:1, max:30, step:1}

if peptide_chain == target_chain:
    raise ValueError("Peptide and target chains must be different!")

print(f"⚙️  Configuration:")
print(f"   Binder chain:       {peptide_chain}")
print(f"   Target chain:       {target_chain}")
print(f"   FastRelax repeats:  {fastrelax_repeats}")
print(f"   Partners string:    {target_chain}_{peptide_chain}")

⚙️  Configuration:
   Binder chain:       B
   Target chain:       A
   FastRelax repeats:  1
   Partners string:    A_B


In [11]:
#@title **1.1 — Run FastRelax + Binding Energy** { display-mode: "form" }
#@markdown > Relaxes each structure with constrained FastRelax, then calculates
#@markdown > binding energy by separating the chains.
#@markdown >
#@markdown > **Runtime:** ~2–5 min per structure on a Colab GPU/CPU.

import pyrosetta
from pyrosetta import *
from pyrosetta.rosetta.core.pack.task import *
from pyrosetta.rosetta.protocols import *
from pyrosetta.rosetta.core.select import *
from pyrosetta.rosetta.protocols.relax import FastRelax
import os, time, csv

pyrosetta.init("-constant_seed -jran 11111 -mute all")

RELAX_DIR = os.path.join(RESULTS_DIR, "01_relaxed")
os.makedirs(RELAX_DIR, exist_ok=True)

def calculate_binding_energy(pose, partners):
    testPose = Pose()
    testPose.assign(pose)
    scorefxn = get_fa_scorefxn()

    # Unbound reference energy
    unbind(testPose, partners)
    native_ub = scorefxn(testPose)
    testPose.assign(pose)

    # Bound vs unbound
    bound = scorefxn(testPose)
    unbind(testPose, partners)
    unbound = scorefxn(testPose)
    binding = bound - unbound
    return native_ub, bound, unbound, binding

def unbind(pose, partners):
    STEP_SIZE = 100
    JUMP = 1
    docking.setup_foldtree(pose, partners, Vector1([1, 0, 0]))
    trans_mover = rigid.RigidBodyTransMover(pose, JUMP)
    trans_mover.step_size(STEP_SIZE)
    trans_mover.apply(pose)

partners = f"{target_chain}_{peptide_chain}"
pdb_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdb")))

print(f"🚀 STAGE 1: FastRelax + Binding Energy ({len(pdb_files)} structures)")
print(f"   Repeats: {fastrelax_repeats} | Partners: {partners}\n")

binding_results = []
t0_total = time.time()

for i, pdb_path in enumerate(pdb_files, 1):
    name = Path(pdb_path).stem
    print(f"  [{i}/{len(pdb_files)}] {name}...", end=" ", flush=True)
    t0 = time.time()

    pose = pose_from_pdb(pdb_path)

    # FastRelax setup
    testPose = Pose()
    testPose.assign(pose)
    mm = MoveMap()
    mm.set_bb(True)
    mm.set_chi(True)

    relax = FastRelax(standard_repeats=fastrelax_repeats)
    relax.set_movemap(mm)
    scorefxn = get_fa_scorefxn()
    relax.set_scorefxn(scorefxn)
    relax.constrain_relax_to_start_coords(True)
    relax.coord_constrain_sidechains(True)

    # Apply relaxation
    relax.apply(testPose)
    relax_pdb = os.path.join(RELAX_DIR, f"{name}_relax.pdb")
    testPose.dump_pdb(relax_pdb)
    relaxPose = pose_from_pdb(relax_pdb)

    # Binding energy
    native_ub, bound, unbound, binding = calculate_binding_energy(relaxPose, partners)

    binding_results.append({
        "Filename": name,
        "relax_pdb": relax_pdb,
        "Bound": bound,
        "Unbound": unbound,
        "Binding_Energy": binding,
    })

    elapsed = time.time() - t0
    print(f"ddG_approx={binding:.1f} REU  ({elapsed:.0f}s)")

total_time = time.time() - t0_total

import pandas as pd
df_binding = pd.DataFrame(binding_results)
be_csv = os.path.join(RESULTS_DIR, "binding_energies.csv")
df_binding.to_csv(be_csv, index=False)

print(f"\n✅ STAGE 1 COMPLETE ({total_time/60:.1f} min)")
print(f"   Relaxed PDBs:     {RELAX_DIR}/")
print(f"   Binding energies: {be_csv}")
display(df_binding[["Filename", "Bound", "Unbound", "Binding_Energy"]].round(2))

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.34+release.de92a3c0dea8a010d372a22025e3e50bd4e2f33f 2026-08-19T16:08:45] retrieved from: http://www.pyrosetta.org
🚀 STAGE 1: FastRelax + Binding Energy (16 structures)
   Repeats: 1 | Partners: A_B

  [1/16] inputs_fpr2_binder_0_model_0.cif_b0_d0_relaxed_rank_001_alphafold2_mu

,Filename,Bound,Unbound,Binding_Energy
0,inputs_fpr2_binder_0_model_0.cif_b0_d0_relaxed...,-874.88,-809.73,-65.15
1,inputs_fpr2_binder_0_model_0.cif_b1_d0_relaxed...,-860.88,-795.14,-65.74
2,inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed...,-847.82,-778.73,-69.09
3,inputs_fpr2_binder_0_model_0.cif_b3_d0_relaxed...,-832.85,-769.71,-63.13
4,inputs_fpr2_binder_0_model_1.cif_b0_d0_relaxed...,-858.36,-794.22,-64.14
5,inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed...,-854.31,-785.13,-69.19
6,inputs_fpr2_binder_0_model_1.cif_b2_d0_relaxed...,-854.19,-783.21,-70.98
7,inputs_fpr2_binder_0_model_1.cif_b3_d0_relaxed...,-846.94,-780.58,-66.36
8,inputs_fpr2_binder_0_model_2.cif_b0_d0_relaxed...,-847.14,-777.36,-69.79
9,inputs_fpr2_binder_0_model_2.cif_b1_d0_relaxed...,-869.54,-798.12,-71.42


---
## 2. Stage 2: Rosetta Physics Filters

Each relaxed structure is evaluated by a panel of physics-based filters from RosettaScripts:

| Filter | Measures | Good value |
|--------|----------|------------|
| **ddG** | Binding free energy (interface stability) | < –30 REU |
| **SAP score** | Spatial Aggregation Propensity (solubility proxy) | < 35 |
| **Contact molecular surface** | Complementarity of interface surfaces | > 300 |
| **Interface buried SASA** | Total interface area buried upon binding | Higher = more contact |
| **BUH** | Buried unsatisfied H-bond donors/acceptors | Lower = better |

The RosettaScripts XML protocol also performs interface minimization before scoring.

In [16]:
#@title **2.0 — Run Rosetta Physics Filters** { display-mode: "form" }
#@markdown > Applies RosettaScripts XML protocol to each relaxed PDB:
#@markdown > interface minimization → ddG → SAP score → SASA → BUH.
#@markdown >
#@markdown > **Runtime:** ~1–3 min per structure.

import pyrosetta
from pyrosetta import *
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects
import os, csv, time

pyrosetta.init("-corrections:beta_nov16 -mute all -ignore_unrecognized_res true -nstruct 1")

FILTER_DIR = os.path.join(RESULTS_DIR, "02_filtered")
os.makedirs(FILTER_DIR, exist_ok=True)

xml = XmlObjects.create_from_string("""
<ROSETTASCRIPTS>

  <SCOREFXNS>
    <ScoreFunction name="sfxn_cart" weights="beta_nov16_cart" >
      <Reweight scoretype="coordinate_constraint" weight="1" />
      <Reweight scoretype="atom_pair_constraint" weight="1" />
      <Reweight scoretype="dihedral_constraint" weight="1" />
      <Reweight scoretype="angle_constraint" weight="1" />
    </ScoreFunction>
  </SCOREFXNS>

  <RESIDUE_SELECTORS>
    <Chain name="chainB" chains="1"/>
    <Chain name="chainA" chains="2"/>
    <Neighborhood name="interface_chA" selector="chainB" distance="14.0" />
    <Neighborhood name="interface_chB" selector="chainA" distance="14.0" />
    <And name="AB_interface" selectors="interface_chA,interface_chB" />
    <Not name="Not_interface" selector="AB_interface" />
  </RESIDUE_SELECTORS>

  <TASKOPERATIONS>
    <ProteinInterfaceDesign name="pack_long" design_chain1="0" design_chain2="0" jump="1" interface_distance_cutoff="15"/>
    <OperateOnResidueSubset name="restrict_to_interface" selector="Not_interface">
      <PreventRepackingRLT/>
    </OperateOnResidueSubset>
  </TASKOPERATIONS>

  <MOVERS>
    <TaskAwareMinMover name="minimize_interface" scorefxn="sfxn_cart" tolerance="0.01" cartesian="true" task_operations="restrict_to_interface" jump="0" />
    <TaskAwareMinMover name="min" scorefxn="sfxn_cart" bb="0" chi="1" task_operations="pack_long" />
  </MOVERS>

  <FILTERS>
    <Ddg name="ddg" threshold="50" jump="1" repeats="5" repack="1" relax_mover="min" confidence="0" scorefxn="sfxn_cart" extreme_value_removal="1" />
    <ContactMolecularSurface name="contact_molecular_surface" distance_weight="0.5" target_selector="chainA" binder_selector="chainB" confidence="0" />
    <Sasa name="interface_buried_sasa" confidence="0"/>
    <Sasa name="interface_hydrophobic_sasa" confidence="0" hydrophobic="True"/>
    <Sasa name="interface_polar_sasa" confidence="0" polar="True"/>
    <BuriedUnsatHbonds name="BUH" scorefxn="sfxn_cart" confidence="0" jump_number="1"/>
  </FILTERS>

  <SIMPLE_METRICS>
    <SapScoreMetric name="sap_score" score_selector="chainA" />
  </SIMPLE_METRICS>

  <PROTOCOLS>
    <Add mover="minimize_interface" />
    <Add filter="ddg" />
    <Add metrics="sap_score" />
    <Add filter="contact_molecular_surface" />
    <Add filter="interface_buried_sasa"/>
    <Add filter="interface_hydrophobic_sasa"/>
    <Add filter="interface_polar_sasa"/>
    <Add filter="BUH"/>
  </PROTOCOLS>

</ROSETTASCRIPTS>
""")

protocol = xml.get_mover("ParsedProtocol")

# Define the fixed header
fixed_header = [
    "Filename", "BUH", "contact_molecular_surface", "ddg", "interface_buried_sasa",
    "interface_hydrophobic_sasa", "interface_polar_sasa", "sap_score", "fa_atr",
    "fa_rep", "fa_sol", "fa_intra_atr_xover4", "fa_intra_rep_xover4",
    "fa_intra_sol_xover4", "lk_ball", "lk_ball_iso", "lk_ball_bridge",
    "lk_ball_bridge_uncpl", "fa_elec", "fa_intra_elec", "hbond_sr_bb",
    "hbond_lr_bb", "hbond_bb_sc", "hbond_sc", "dslf_fa13", "atom_pair_constraint",
    "coordinate_constraint", "angle_constraint", "dihedral_constraint", "omega",
    "fa_dun_dev", "fa_dun_rot", "fa_dun_semi", "p_aa_pp", "hxl_tors", "ref",
    "rama_prepro", "cart_bonded", "total_score"
]

relaxed_pdbs = sorted(glob.glob(os.path.join(RELAX_DIR, "*_relax.pdb")))

print(f"🚀 STAGE 2: Rosetta Physics Filters ({len(relaxed_pdbs)} structures)\n")

all_scores = []
t0_total = time.time()

for i, pdb_path in enumerate(relaxed_pdbs, 1):
    name = Path(pdb_path).stem
    print(f"  [{i}/{len(relaxed_pdbs)}] {name}...", end=" ", flush=True)
    t0 = time.time()

    pose = pose_from_pdb(pdb_path)
    protocol.apply(pose)
    score_end = pose.scores

    # Collect scores
    complete_data = {key: score_end.get(key, "") for key in fixed_header}
    complete_data["Filename"] = name
    all_scores.append(complete_data)

    # Save individual CSV
    indiv_csv = os.path.join(FILTER_DIR, f"{name}_filters.csv")
    with open(indiv_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fixed_header)
        w.writeheader()
        w.writerow(complete_data)

    # Save output PDB
    pose.dump_pdb(os.path.join(FILTER_DIR, f"{name}_output.pdb"))

    elapsed = time.time() - t0
    ddg_val = complete_data.get("ddg", "N/A")
    sap_val = complete_data.get("sap_score", "N/A")
    print(f"ddG={ddg_val}  SAP={sap_val}  ({elapsed:.0f}s)")

total_time = time.time() - t0_total

# Combined CSV
combined_csv = os.path.join(RESULTS_DIR, "combined_scores.csv")
with open(combined_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fixed_header)
    w.writeheader()
    w.writerows(all_scores)

import pandas as pd
df_scores = pd.read_csv(combined_csv)

print(f"\n✅ STAGE 2 COMPLETE ({total_time/60:.1f} min)")
print(f"   Combined scores: {combined_csv}")
print(f"\n📊 Key metrics summary:")
key_cols = ["Filename", "ddg", "sap_score", "contact_molecular_surface",
            "interface_buried_sasa", "BUH", "total_score"]
display(df_scores[[c for c in key_cols if c in df_scores.columns]].round(2))

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.34+release.de92a3c0dea8a010d372a22025e3e50bd4e2f33f 2026-08-19T16:08:45] retrieved from: http://www.pyrosetta.org
🚀 STAGE 2: Rosetta Physics Filters (16 structures)

  [1/16] inputs_fpr2_binder_0_model_0.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax.

/tmp/ipykernel_6453/1424574688.py:106: DeprecationWarning: The `Pose.scores` dictionary is deprecated and may be aliased to the `Pose.cache` dictionary in a future release. Prefer to use the `Pose.cache` dictionary instead.
  score_end = pose.scores


ddG=-58.486083984375  SAP=32.54711151123047  (20s)
  [3/16] inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax... ddG=-66.93637084960938  SAP=29.950756072998047  (19s)
  [4/16] inputs_fpr2_binder_0_model_0.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax... ddG=-54.311336517333984  SAP=33.74432373046875  (22s)
  [5/16] inputs_fpr2_binder_0_model_1.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax... ddG=-59.269832611083984  SAP=35.91670227050781  (18s)
  [6/16] inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax... ddG=-65.02257537841797  SAP=27.569416046142578  (21s)
  [7/16] inputs_fpr2_binder_0_model_1.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax... ddG=-54.66823959350586  SAP=27.58812713623047  (21s)
  [8/16] inputs_fpr2_binder_0_model_1.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax

,Filename,ddg,sap_score,contact_molecular_surface,interface_buried_sasa,BUH,total_score
0,inputs_fpr2_binder_0_model_0.cif_b0_d0_relaxed...,-35.83,30.09,558.34,2109.77,30.0,-277.59
1,inputs_fpr2_binder_0_model_0.cif_b1_d0_relaxed...,-58.49,32.55,589.38,2150.39,31.0,-315.21
2,inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed...,-66.94,29.95,595.18,2056.62,34.0,-549.12
3,inputs_fpr2_binder_0_model_0.cif_b3_d0_relaxed...,-54.31,33.74,610.09,2219.83,27.0,-118.79
4,inputs_fpr2_binder_0_model_1.cif_b0_d0_relaxed...,-59.27,35.92,546.58,2133.91,28.0,-110.27
5,inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed...,-65.02,27.57,576.92,2140.66,28.0,-210.43
6,inputs_fpr2_binder_0_model_1.cif_b2_d0_relaxed...,-54.67,27.59,572.09,2192.97,32.0,-1.86
7,inputs_fpr2_binder_0_model_1.cif_b3_d0_relaxed...,-58.84,30.15,563.85,2166.56,32.0,-236.19
8,inputs_fpr2_binder_0_model_2.cif_b0_d0_relaxed...,-62.73,26.77,630.13,2252.22,35.0,-617.69
9,inputs_fpr2_binder_0_model_2.cif_b1_d0_relaxed...,-61.72,25.92,610.19,2292.65,35.0,-202.70


---
## 3. Stage 3: Filter Best Designs

Apply physics-based cutoffs to select the most promising candidates.
Default thresholds (adjustable via sliders):

| Metric | Cutoff | Rationale |
|--------|--------|-----------|
| ddG | < –30 REU | Strong predicted binding |
| SAP score | < 35 | Good predicted solubility |
| Contact molecular surface | > 300 | Sufficient interface complementarity |

In [17]:
#@title **3.0 — Filter Best Designs** { display-mode: "form" }

#@markdown ### Physics Filter Cutoffs
ddg_cutoff     = -30   #@param {type:"slider", min:-100, max:0, step:5}
sap_cutoff     = 35    #@param {type:"slider", min:0, max:100, step:5}
cms_cutoff     = 300   #@param {type:"slider", min:0, max:1000, step:50}

import pandas as pd

df_scores = pd.read_csv(os.path.join(RESULTS_DIR, "combined_scores.csv"))

print(f"🔍 STAGE 3: Filtering (ddG < {ddg_cutoff}, SAP < {sap_cutoff}, CMS > {cms_cutoff})\n")

# Convert to numeric where needed
for col in ["ddg", "sap_score", "contact_molecular_surface"]:
    df_scores[col] = pd.to_numeric(df_scores[col], errors="coerce")

# Apply filters
filtered_df = df_scores[
    (df_scores["ddg"] < ddg_cutoff) &
    (df_scores["sap_score"] < sap_cutoff) &
    (df_scores["contact_molecular_surface"] > cms_cutoff)
].copy()

print(f"  Total structures:  {len(df_scores)}")
print(f"  Passed filters:    {len(filtered_df)}/{len(df_scores)}")

if len(filtered_df) > 0:
    output_df = filtered_df[
        ["Filename", "ddg", "sap_score", "contact_molecular_surface",
         "interface_buried_sasa", "BUH", "total_score"]
    ].sort_values("ddg")

    filtered_csv = os.path.join(RESULTS_DIR, "filtered_scores.csv")
    output_df.to_csv(filtered_csv, index=False)

    print(f"\n  🏆 Best designs (sorted by ddG):")
    display(
        output_df.style.background_gradient(subset=["ddg"], cmap="RdYlGn_r")
                       .background_gradient(subset=["sap_score"], cmap="RdYlGn_r")
                       .background_gradient(subset=["contact_molecular_surface"], cmap="YlGn")
                       .format(precision=2)
    )
    print(f"\n✅ Filtered results saved: {filtered_csv}")
else:
    print("\n  ⚠️  No designs passed all filters.")
    print("  Try relaxing the cutoffs (less negative ddG, higher SAP, lower CMS).")
    print("\n  Showing top 5 by ddG regardless:")
    output_df = df_scores.nsmallest(5, "ddg")[
        ["Filename", "ddg", "sap_score", "contact_molecular_surface",
         "interface_buried_sasa", "BUH", "total_score"]
    ]
    display(output_df.round(2))
    filtered_csv = os.path.join(RESULTS_DIR, "filtered_scores.csv")
    output_df.to_csv(filtered_csv, index=False)

🔍 STAGE 3: Filtering (ddG < -30, SAP < 35, CMS > 300)

  Total structures:  16
  Passed filters:    13/16

  🏆 Best designs (sorted by ddG):


,Filename,ddg,sap_score,contact_molecular_surface,interface_buried_sasa,BUH,total_score
12,inputs_fpr2_binder_0_model_3.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-75.85,31.37,682.62,2348.81,35.00,-613.74
2,inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-66.94,29.95,595.18,2056.62,34.00,-549.12
5,inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-65.02,27.57,576.92,2140.66,28.00,-210.43
15,inputs_fpr2_binder_0_model_3.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-64.11,27.03,642.93,2585.27,30.00,-555.44
8,inputs_fpr2_binder_0_model_2.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-62.73,26.77,630.13,2252.22,35.00,-617.69
9,inputs_fpr2_binder_0_model_2.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-61.72,25.92,610.19,2292.65,35.00,-202.70
10,inputs_fpr2_binder_0_model_2.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-58.86,28.66,577.04,2119.58,40.00,-338.53
7,inputs_fpr2_binder_0_model_1.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-58.84,30.15,563.85,2166.56,32.00,-236.19
1,inputs_fpr2_binder_0_model_0.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-58.49,32.55,589.38,2150.39,31.00,-315.21
11,inputs_fpr2_binder_0_model_2.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax,-56.40,31.21,585.02,2123.75,33.00,-328.79



✅ Filtered results saved: /content/pyrosetta_filters/results/filtered_scores.csv


---
## 4. 3D Visualization of Best Designs

Interactive 3D views of the structures that passed the physics filters.
- 🌈 **Rainbow**: Binder chain, colored N→C
- 🔵 **Light blue**: Target protein

In [19]:
#@title **4.0 — Visualize Best Designs (3D)** { display-mode: "form" }
#@markdown > Shows interactive 3D views of the structures that passed physics filters.

#@markdown ### How many to show?
n_show = 4  #@param {type:"slider", min:1, max:12, step:1}

import py3Dmol
import pandas as pd
from pathlib import Path

filtered_csv = os.path.join(RESULTS_DIR, "filtered_scores.csv")
if not os.path.isfile(filtered_csv):
    print("⚠️  No filtered_scores.csv found. Run the filter cell first.")
else:
    df_filtered = pd.read_csv(filtered_csv)
    n_show = min(n_show, len(df_filtered))

    print(f"🏆 Top {n_show} designs by physics filters:\n")

    for rank, (_, row) in enumerate(df_filtered.head(n_show).iterrows(), 1):
        design_name = row["Filename"]

        # Look for the PDB in filter output, then relaxed, then input
        pdb_path = None
        for search_dir in [FILTER_DIR, RELAX_DIR, INPUT_DIR]:
            candidates = sorted(glob.glob(os.path.join(search_dir, "*.pdb")))
            match = [p for p in candidates if design_name in Path(p).stem]
            if match:
                pdb_path = match[0]
                break

        if not pdb_path:
            print(f"  Design {rank}: {design_name} — PDB not found, skipping")
            continue

        ddg_val = row.get("ddg", float("nan"))
        sap_val = row.get("sap_score", float("nan"))
        cms_val = row.get("contact_molecular_surface", float("nan"))
        print(f"  Design {rank}: {design_name}")
        print(f"    ddG={ddg_val:.1f}  SAP={sap_val:.1f}  CMS={cms_val:.0f}")

        with open(pdb_path) as f:
            pdb_data = f.read()

        v = py3Dmol.view(width=500, height=350)
        v.addModel(pdb_data, "pdb")
        v.setStyle({"chain": peptide_chain}, {"cartoon": {"color": "spectrum"}})
        v.setStyle({"chain": target_chain}, {"cartoon": {"color": "lightblue"}})
        v.setBackgroundColor("white")
        v.zoomTo()
        v.show()
        print()

🏆 Top 4 designs by physics filters:

  Design 1: inputs_fpr2_binder_0_model_3.cif_b0_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax
    ddG=-75.8  SAP=31.4  CMS=683


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


  Design 2: inputs_fpr2_binder_0_model_0.cif_b2_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax
    ddG=-66.9  SAP=30.0  CMS=595


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


  Design 3: inputs_fpr2_binder_0_model_1.cif_b1_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax
    ddG=-65.0  SAP=27.6  CMS=577


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


  Design 4: inputs_fpr2_binder_0_model_3.cif_b3_d0_relaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000_relax
    ddG=-64.1  SAP=27.0  CMS=643


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 5. Download Results

Package all Rosetta outputs (relaxed PDBs, filter scores, filtered candidates) into a ZIP.

In [20]:
#@title **5.0 — Package & Download Results** { display-mode: "form" }
#@markdown > Creates a ZIP with relaxed PDBs, all scores, and filtered candidates.

import zipfile

zip_out = "/content/pyrosetta_filter_results.zip"

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    # Binding energies
    be_csv = os.path.join(RESULTS_DIR, "binding_energies.csv")
    if os.path.isfile(be_csv):
        zf.write(be_csv, "binding_energies.csv")

    # Combined + filtered scores
    for csv_name in ["combined_scores.csv", "filtered_scores.csv"]:
        csv_path = os.path.join(RESULTS_DIR, csv_name)
        if os.path.isfile(csv_path):
            zf.write(csv_path, csv_name)

    # Relaxed PDBs
    for pdb in sorted(glob.glob(os.path.join(RELAX_DIR, "*.pdb"))):
        zf.write(pdb, f"01_relaxed/{os.path.basename(pdb)}")

    # Filter output PDBs
    for pdb in sorted(glob.glob(os.path.join(FILTER_DIR, "*.pdb"))):
        zf.write(pdb, f"02_filter_output/{os.path.basename(pdb)}")

print(f"📦 {zip_out} ({os.path.getsize(zip_out)/1024:.1f} KB)")

from google.colab import files
files.download(zip_out)
print("✅ Download started!")

📦 /content/pyrosetta_filter_results.zip (3415.0 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


---
## 📊 Pipeline Summary

| Stage | Tool | Output |
|-------|------|--------|
| 1 | **FastRelax** | Relaxed PDBs + binding energies |
| 2 | **RosettaScripts Filters** | ddG, SAP, CMS, SASA, BUH scores |
| 3 | **Score Filtering** | Top candidates by physics cutoffs |
| 4 | **py3Dmol** | 3D visualization of best designs |

**Input:** `*_designs.zip` from the RFdiffusion3 Workshop notebook.

**Key output files:**
- `binding_energies.csv` — FastRelax binding energies
- `combined_scores.csv` — Full Rosetta score breakdown
- `filtered_scores.csv` — Candidates passing physics cutoffs

**References:**
- Alford et al. (2017). *The Rosetta All-Atom Energy Function*. J. Chem. Theory Comput.
- Norn et al. (2021). *Protein sequence design by conformational landscape optimization*. PNAS.